# 框架运行时、数据与性能补充线 · 第 5/8 课：数据分片、Shuffle 与精确 Resume

> 状态：**参考答案版**  
> 逐课通过制：未完成代码、问答与边界解释前，不进入下一课。

## 本课目标与完成标准

本课产出：实现无重复的全局样本位置分配，并说明 epoch、seed、world-size 变化下的恢复语义。

通过要求：唯一代码填空题通过给定检查；Q1～Q3 都给出因果链；能指出一个正确性边界和一个性能/运营取舍；总分至少 8/10。

## 与既有六条主线的边界

`train/` 把 batch 当数字；本课关注样本如何在 rank/worker 间唯一分配、何时重复/丢失，以及 checkpoint 如何恢复数据位置。

前置：Python、PyTorch、train 第 1～5 课、CUDA 基础。本课只补 AI Infra 缺口，不重复已经完成的 kernel 或并行算法推导。

## 核心心智模型

### 它是什么、解决什么问题

先用 `(seed,epoch)` 生成全局顺序，再按 global position 分到 rank；worker 在 rank 内继续切分。resume 至少保存 epoch、全局 cursor、sampler RNG 和数据版本。

### 数据与控制如何流动

数据快照和 `(seed, epoch)` 确定全局样本序列；global position 先映射到 rank，再映射到 worker。只有消费提交后才推进可恢复 cursor，checkpoint 把 cursor 与模型状态绑定。

### 正确性条件与常见误区

`DistributedSampler.set_epoch()` 必须每 epoch 调用；IterableDataset 多 worker 需自行分片。world size 变化时按 rank-local cursor 恢复会重复或跳过。

### 性能、成本与工程取舍

严格不重复恢复需要全局位置和稳定数据快照，状态更复杂；允许少量重复可简化故障恢复，但要评估优化偏差和账单。

## 具体演示

全局位置 g 按 `g % world_size` 分 rank：world=4 时位置 10 属 rank2。若只保存 rank2 已读 3 条，world 改 8 后语义不再唯一。

请先独立复述“输入 → 状态变化 → 输出/指标”，再开始填空。

## 实践任务：唯一代码填空题

补齐全局位置到 rank 的确定性分配，并返回本 rank 的局部序号。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
def owner_and_local_index(global_position, world_size):
    if global_position < 0 or world_size <= 0:
        raise ValueError("invalid shard position")
    # TODO：round-robin 分片，owner=g%P，local=g//P。
    return ______

assert owner_and_local_index(10, 4) == (2, 2)
assert owner_and_local_index(0, 8) == (0, 0)


### 检查方法

运行本单元格，所有断言必须通过；再补一个边界输入并解释预期。

### Q1

忘记 `DistributedSampler.set_epoch(epoch)` 会怎样？

**你的答案：**


### Q2

IterableDataset + num_workers=8 为什么可能把数据重复 8 次？

**你的答案：**


### Q3

弹性恢复 world size 改变时，怎样减少重复/丢失？

**你的答案：**


## 评分规则

- 代码 4 分：正常输入 2 分、边界输入 1 分、解释 1 分；
- Q1～Q3 各 2 分；
- 一票否决：混淆测量与推测、忽略租户/请求隔离、用平均值掩盖尾延迟、删除失败路径。

## 参考答案（仅 answer 分支）

先独立完成。核对后请改变一个规模或故障条件重新推演。

In [ ]:
def owner_and_local_index(global_position, world_size):
    if global_position < 0 or world_size <= 0:
        raise ValueError("invalid shard position")
    return global_position % world_size, global_position // world_size

assert owner_and_local_index(10, 4) == (2, 2)
assert owner_and_local_index(0, 8) == (0, 0)


### Q1 参考答案

各 epoch 会重复相同伪随机排列，rank 间仍可能不重叠，但跨 epoch 的 shuffle 没变化，降低数据随机性。不能通过单 epoch 无重复测试发现。

### Q2 参考答案

每个 worker 拥有 dataset 副本；若 iterator 不根据 worker_info/rank 切数据，每个副本从同一源头开始。DataLoader 无法替用户代码猜测分片语义。

### Q3 参考答案

保存稳定数据快照上的全局 cursor/已提交范围，而不是每 rank 私有 offset；重建新分片并从全局边界继续。分布式预取中的未提交样本需要明确 at-least-once 或丢弃策略。

## 参考资料

- [torch.utils.data](https://docs.pytorch.org/docs/stable/data.html)
- [Distributed Checkpoint](https://docs.pytorch.org/docs/stable/distributed.checkpoint.html)

API 与平台能力会演进；部署前应按目标版本重新核对。